In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Functions are needed to prepare the data, train the model with the data
# and evaluate the model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("imports reuissis")

imports reuissis


In [ ]:
#Loading the data

train_perf = pd.read_csv('/content/drive/MyDrive/Zindi_Loan/trainperf.csv')
train_demo = pd.read_csv('/content/drive/MyDrive/Zindi_Loan/traindemographics.csv')
train_prev = pd.read_csv('/content/drive/MyDrive/Zindi_Loan/trainprevloans.csv')

test_perf = pd.read_csv('/content/drive/MyDrive/Zindi_Loan/testperf.csv')
test_demo = pd.read_csv('/content/drive/MyDrive/Zindi_Loan/testdemographics.csv')
test_prev = pd.read_csv('/content/drive/MyDrive/Zindi_Loan/testprevloans.csv')

print("Données chargées")
print(f"Train:{train_perf.shape} clients")
print(f"Test:{test_perf.shape} clients")


Données chargées
Train:(4368, 10) clients
Train:(1450, 9) clients


In [ ]:
#Exploring the target column
#This allows us to see balance in data we want to train the model with.
#Here, we have more good than bad data which means that the model will
#be more biased to predicting "good"
#To improve the model accuracy there are a few techniques
#Option 1: Fix the data (resampling)
#Option 2: Tell the model about the imbalance with class weight balance
#Option 3: Use better metrics

print(train_perf['good_bad_flag'].value_counts())



good_bad_flag
Good    3416
Bad      952
Name: count, dtype: int64


In [ ]:
#Here we are checking some stats about the train_prev table per loaner
#The number of loans they took
#The average amount of their previous loans
#the average terms_days

train_prev_agg = train_prev.groupby('customerid').agg(
    nb_prev_loans = ('systemloanid', 'count'),
    avg_prev_amount = ('loanamount', 'mean'),
    avg_prev_termdays = ('termdays', 'mean')
).reset_index()

#This line of code allows to display all of the columns
pd.set_option('display.max_columns', None)
print(train_prev_agg.head(10))


                         customerid  nb_prev_loans  avg_prev_amount  \
0  8a1088a0484472eb01484669e3ce4e0b              1     10000.000000   
1  8a1a1e7e4f707f8b014f797718316cad              4     17500.000000   
2  8a1a32fc49b632520149c3b8fdf85139              7     12857.142857   
3  8a1eb5ba49a682300149c3c068b806c7              8     16250.000000   
4  8a1edbf14734127f0147356fdb1b1eb2              2     10000.000000   
5  8a26bd845089f1d7015090b1d6f53bad              9     13333.333333   
6  8a2a81a74ce8c05d014cfb32a0da1049             11     18181.818182   
7  8a2ac4745091002b0150a144bcbe58b7              7     22857.142857   
8  8a2ad9ce4c453e06014c4b3175e52407              1     20000.000000   
9  8a33a06e4a5075c2014a5295aa0c2224              8     12500.000000   

   avg_prev_termdays  
0          15.000000  
1          37.500000  
2          19.285714  
3          33.750000  
4          22.500000  
5          26.666667  
6          30.000000  
7          42.857143  
8          

In [ ]:
#I just realised that putting print in a different cell is a much better option
print(train_prev_agg.head (10))

                         customerid  nb_prev_loans  avg_prev_amount  \
0  8a1088a0484472eb01484669e3ce4e0b              1     10000.000000   
1  8a1a1e7e4f707f8b014f797718316cad              4     17500.000000   
2  8a1a32fc49b632520149c3b8fdf85139              7     12857.142857   
3  8a1eb5ba49a682300149c3c068b806c7              8     16250.000000   
4  8a1edbf14734127f0147356fdb1b1eb2              2     10000.000000   
5  8a26bd845089f1d7015090b1d6f53bad              9     13333.333333   
6  8a2a81a74ce8c05d014cfb32a0da1049             11     18181.818182   
7  8a2ac4745091002b0150a144bcbe58b7              7     22857.142857   
8  8a2ad9ce4c453e06014c4b3175e52407              1     20000.000000   
9  8a33a06e4a5075c2014a5295aa0c2224              8     12500.000000   

   avg_prev_termdays  
0          15.000000  
1          37.500000  
2          19.285714  
3          33.750000  
4          22.500000  
5          26.666667  
6          30.000000  
7          42.857143  
8          

In [ ]:
#Let's join the 3 "train" tables on Customerid

train = train_perf.merge(train_prev, on ='customerid', how='left')
train = train.merge(train_demo, on ='customerid', how='left')


In [ ]:
train.head(10)

,customerid,systemloanid_x,loannumber_x,approveddate_x,creationdate_x,loanamount_x,totaldue_x,termdays_x,referredby_x,good_bad_flag,systemloanid_y,loannumber_y,approveddate_y,creationdate_y,loanamount_y,totaldue_y,termdays_y,closeddate,referredby_y,firstduedate,firstrepaiddate,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301682320.0,2.0,2016-08-15 18:22:40.000000,2016-08-15 17:22:32.000000,10000.0,13000.0,30.0,2016-09-01 16:06:48.000000,NaN,2016-09-14 00:00:00.000000,2016-09-01 15:51:43.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
1,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301883808.0,9.0,2017-04-28 18:39:07.000000,2017-04-28 17:38:53.000000,10000.0,13000.0,30.0,2017-05-28 14:44:49.000000,NaN,2017-05-30 00:00:00.000000,2017-05-26 00:00:00.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
2,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301831714.0,8.0,2017-03-05 10:56:25.000000,2017-03-05 09:56:19.000000,20000.0,23800.0,30.0,2017-04-26 22:18:56.000000,NaN,2017-04-04 00:00:00.000000,2017-04-26 22:03:47.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
3,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301923941.0,10.0,2017-06-01 13:34:30.000000,2017-06-01 12:34:21.000000,20000.0,24500.0,30.0,2017-06-25 15:24:06.000000,NaN,2017-07-03 00:00:00.000000,2017-06-25 15:13:56.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
4,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301954468.0,11.0,2017-06-28 10:58:34.000000,2017-06-28 09:58:25.000000,20000.0,24500.0,30.0,2017-07-25 08:14:36.000000,NaN,2017-07-31 00:00:00.000000,2017-07-25 08:04:27.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
5,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301777989.0,6.0,2016-12-21 19:03:40.000000,2016-12-21 11:02:54.000000,20000.0,23800.0,30.0,2017-02-28 13:20:29.000000,NaN,2017-01-20 00:00:00.000000,2017-02-28 13:05:11.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
6,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301739329.0,4.0,2016-10-07 19:00:48.000000,2016-10-07 18:00:37.000000,20000.0,24500.0,30.0,2016-11-07 08:29:43.000000,NaN,2016-11-07 00:00:00.000000,2016-11-07 08:14:34.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
7,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301828139.0,7.0,2017-02-28 13:25:25.000000,2017-02-28 12:25:18.000000,20000.0,23800.0,30.0,2017-03-01 18:25:25.000000,NaN,2017-03-30 00:00:00.000000,2017-03-01 18:10:14.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
8,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,301675247.0,1.0,2016-08-09 22:57:01.000000,2016-08-09 21:56:47.000000,10000.0,13000.0,30.0,2016-08-15 08:49:18.000000,NaN,2016-09-08 00:00:00.000000,2016-08-15 08:34:14.000000,1972-01-15 00:00:00.000000,Other,3.43201,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
9

In [ ]:
#The column good_bad_flag that we are training the model to predict is in a
#format text. It must be converted into 1 and 0 which is a binary language the
#model can understand

#Still having an hard time wrapping my mind around this line of code
train['target'] = (train['good_bad_flag'] == 'Good').astype(int)

#Now, we are selecting useful columns as features for the model

